# Software Engineering Refresh — Hands-On

**Software Engineering · Week 01**

Offline notebook: stdlib-only simulations of boundaries, patterns, structured logs, and tests.

## 0. Setup: a tiny domain

In [ ]:
from dataclasses import dataclass
from typing import Protocol
import json, logging, io

@dataclass(frozen=True)
class Ticket:
    customer: str
    severity: int
    text: str

tickets = [Ticket("acme", 5, "payment down"), Ticket("beta", 2, "ui typo")]
print(tickets)

## 1. Strategy + Dependency Inversion

In [ ]:
class PriorityPolicy(Protocol):
    def score(self, ticket: Ticket) -> int: ...

class SeverityPolicy:
    def score(self, ticket):
        return ticket.severity * 10

class EnterpriseBoostPolicy:
    def __init__(self, enterprise_customers):
        self.enterprise = set(enterprise_customers)
    def score(self, ticket):
        return ticket.severity * 10 + (50 if ticket.customer in self.enterprise else 0)

class Router:
    def __init__(self, policy: PriorityPolicy):
        self.policy = policy
    def route_order(self, tickets):
        return sorted(tickets, key=self.policy.score, reverse=True)

print([t.customer for t in Router(EnterpriseBoostPolicy(["beta"])).route_order(tickets)])

## 2. Adapter boundary around infrastructure

In [ ]:
class TicketStore(Protocol):
    def save(self, ticket: Ticket) -> str: ...

class InMemoryTicketStore:
    def __init__(self):
        self.rows = {}
    def save(self, ticket):
        key = f"T{len(self.rows)+1}"
        self.rows[key] = ticket
        return key

store = InMemoryTicketStore()
print(store.save(tickets[0]), store.rows)

## 3. Structured logging at the application boundary

In [ ]:
stream = io.StringIO()
logger = logging.getLogger("se_week_01_nb")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler(stream)
handler.setFormatter(logging.Formatter("%(message)s"))
logger.handlers[:] = [handler]

def log_event(event, **fields):
    logger.info(json.dumps({"event": event, **fields}, sort_keys=True))

log_event("ticket.routed", customer="acme", score=50, request_id="req-7")
print(stream.getvalue().strip())

## 4. Error handling: wrap infrastructure, preserve cause

In [ ]:
class StoreUnavailable(Exception):
    pass

class FlakyStore:
    def save(self, ticket):
        raise TimeoutError("simulated timeout")

def submit(ticket, store):
    try:
        return store.save(ticket)
    except TimeoutError as exc:
        log_event("ticket.store_unavailable", customer=ticket.customer, error_type=type(exc).__name__)
        raise StoreUnavailable("ticket store unavailable") from exc

try:
    submit(tickets[0], FlakyStore())
except StoreUnavailable as exc:
    print(type(exc).__name__, "cause=", type(exc.__cause__).__name__)

## 5. Fast tests pin behavior

In [ ]:
def test_enterprise_boost_changes_order():
    ordered = Router(EnterpriseBoostPolicy(["beta"])).route_order(tickets)
    assert ordered[0].customer == "beta"

def test_store_assigns_stable_ids():
    s = InMemoryTicketStore()
    assert s.save(tickets[0]) == "T1"
    assert s.save(tickets[1]) == "T2"

for test in [test_enterprise_boost_changes_order, test_store_assigns_stable_ids]:
    test()
print("2 fast tests passed")

## 6. A tiny CI gate simulation

In [ ]:
checks = {"format": True, "unit_tests": True, "type_smoke": True, "notebook_executes": True}
if not all(checks.values()):
    raise SystemExit("CI failed")
print("CI gate green:", checks)

## Exercises
1. Add a second routing policy for SLA deadlines.
2. Write contract tests that every `TicketStore` implementation must pass.
3. Add a correlation/request id to every log event.
4. Decide which checks belong pre-commit, in PR CI, and in deployment.

## Links
- Literature note: `02 Literature Notes/Software Engineering/Software Engineering Refresh`
- Snippets: `04 Code Snippets/Software Engineering/SE Week 01 SOLID Strategy Boundary Example`, `.../SE Week 01 Structured Logging and Error Boundary`
- MOC: `06 Maps of Content/Software Engineering Concepts`